# KnowledgeHub RAG v0.6.3 – Confidence-Gated RAG with a Stronger LLM

## Objective

v0.6.3 addresses the two biggest limitations carried over from v0.6.1/v0.6.2:

1. **Weak generation model.** TinyLlama (1.1B) is replaced with a much stronger
   instruction-tuned open-source model, using proper chat-template formatting
   instead of raw text completion.
2. **No real "don't know" mechanism.** Previously, refusing to answer
   out-of-document questions relied entirely on the LLM following a prompt
   instruction — and the v0.6.2 eval run showed it doesn't reliably do that
   (the optimizer/GPU questions were answered with invented details even
   though the prompt explicitly told the model not to). v0.6.3 adds a
   **retrieval-confidence gate**: if nothing relevant was actually retrieved,
   the system returns the "don't know" message *without ever calling the LLM*,
   so refusal no longer depends on the model's good behavior.

The hybrid retrieval pipeline (FAISS + BM25 + query expansion + RRF + context
expansion) and conversation memory from v0.6.1/v0.6.2 are unchanged.

## What changed in this version

- **Model swap:** TinyLlama-1.1B → Qwen2.5-3B-Instruct (see Cell: "Load the LLM").
- **Chat-template prompting** instead of raw f-string completion, which
  instruct models are actually trained to follow.
- **Retrieval confidence gate (`answer_question`)**: computes a confidence
  score from retrieval before generating anything; below threshold, skips
  the LLM and returns the standard "not found" message.
- **Threshold calibration cell**: instead of guessing `CONFIDENCE_THRESHOLD`,
  a new cell runs retrieval confidence over your existing `gold_eval.json`
  and prints the score distribution split by answerable/unanswerable, so you
  pick a threshold based on real numbers.
- **Eval pipeline and interactive loop now run through `answer_question`**,
  so the gold evaluation actually tests the don't-know mechanism, not just
  the raw generator.
- **Decoding bug fix:** answers are now decoded from the newly generated
  token IDs directly, instead of slicing the decoded prompt string by
  character length — the old approach can silently misalign when the
  tokenizer's re-encoding of the prompt doesn't match the original string
  exactly (e.g. inserted special tokens).

## Known limitations carried into v0.7

- Confidence threshold is a single global number; some question types may
  need different thresholds (this is a reasonable v0.7+ refinement, not a
  blocker).
- Still single-document-only evaluation coverage — a second, differently
  formatted PDF should be tested before assuming this generalizes (see the
  pre-Streamlit checklist).
- Still notebook/Colab-based; the next step is porting into a script-based
  service ahead of the Streamlit UI.


In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install -q rank-bm25
!pip install -q langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
import os
import time
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from rank_bm25 import BM25Okapi

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 1 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf


In [ ]:
def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        extracted = page.extract_text()

        if extracted:

            pages.append(
                {
                    "page": page_number,
                    "text": extracted
                }
            )

    return pages


documents = []

for pdf in pdf_files:

    pages = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "pages": pages
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 1 document(s).


In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================================
# Heading Detection
# ==========================================================

def is_heading(line):

    line = line.strip()

    if len(line) < 2:
        return False

    # 1
    # 1.2
    # 2.3.4
    if re.match(r"^\d+(\.\d+)*\s+[A-Z]", line):
        return True

    # ALL CAPS
    if line.isupper() and len(line.split()) <= 8:
        return True

    # Markdown
    if line.startswith("#"):
        return True

    # Very short title
    if len(line.split()) <= 8 and line.endswith(":"):
        return True

    return False


# ==========================================================
# Split page into sections using headings
# ==========================================================

def split_into_sections(text):

    lines = text.split("\n")

    sections = []

    current = []

    for line in lines:

        if is_heading(line):

            if current:
                sections.append("\n".join(current).strip())

            current = [line]

        else:

            current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    return sections


# ==========================================================
# Recursive splitter
# ==========================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


# ==========================================================
# Final Chunking Function
# ==========================================================

def chunk_text(text):

    sections = split_into_sections(text)

    final_chunks = []

    for section in sections:

        if len(section) <= 900:

            final_chunks.append(section)

        else:

            final_chunks.extend(
                splitter.split_text(section)
            )

    return final_chunks

In [ ]:
all_chunks = []

chunk_id = 0

for document in documents:

    for page in document["pages"]:

        sections = split_into_sections(page["text"])

        for section in sections:

            # -----------------------------
            # Extract heading
            # -----------------------------
            lines = section.split("\n")

            heading = ""

            if lines and is_heading(lines[0]):
                heading = lines[0].strip()

            # -----------------------------
            # Split section if needed
            # -----------------------------
            if len(section) <= 900:

                section_chunks = [section]

            else:

                section_chunks = splitter.split_text(section)

            # -----------------------------
            # Save chunks
            # -----------------------------
            for chunk in section_chunks:

                all_chunks.append(
                    {
                        "chunk_id": chunk_id,
                        "document": document["filename"],
                        "page": page["page"],
                        "section": heading,
                        "text": chunk
                    }
                )

                chunk_id += 1

print(f"Created {len(all_chunks)} chunks.")

Created 112 chunks.


In [ ]:
print("=" * 80)

print("FIRST 10 CHUNKS")

print("=" * 80)

for chunk in all_chunks[:10]:

    print(f"\nChunk ID : {chunk['chunk_id']}")

    print(f"Page     : {chunk['page']}")


    print("-" * 80)

    print(chunk["text"][:400])

    print()

FIRST 10 CHUNKS

Chunk ID : 0
Page     : 1
--------------------------------------------------------------------------------
Machine Learning Classification of Binary Neutron
Star Remnants Using Gravitational Wave Data
Surendaranath Kanniyappan
Dr. Michalis Agathos
Abstract
Binary neutron star (BNS) mergers are among the most energetic cosmic events,
producing gravitational waves (GWs), electromagnetic (EM) counterparts, and potentially
neutrinos. These mergers provide an unparalleled opportunity to study supranuclear m


Chunk ID : 1
Page     : 1
--------------------------------------------------------------------------------
hypermassive neutron star (short- or long-lived HMNS), or remains stable.
Direct detection of postmerger GW signals remains challenging due to their high
frequency nature (≳ 1 kHz) and the sensitivity limits of current interferometers. Therefore,
predicting remnant outcomes from inspiral parameters—total mass Mtot, mass ratio q, tidal
deformability ˜Λ, and effecti

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(112, 384)


In [ ]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

112


In [ ]:

tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print(" BM25 Index Built")

 BM25 Index Built


In [ ]:
QUERY_EXPANSION = {

    "algorithm": [
        "model",
        "classifier",
        "GBDT",
        "Gradient Boosted Decision Tree"
    ],

    "gbdt": [
        "Gradient Boosted Decision Tree",
        "gradient boosting"
    ],

    "classifier": [
        "classification model",
        "machine learning model"
    ],

    "accuracy": [
        "performance",
        "evaluation",
        "MCC"
    ],

    "dataset": [
        "training data",
        "simulation dataset"
    ],

    "method": [
        "approach",
        "framework"
    ]
}


def expand_query(query):

    expanded = query

    query_lower = query.lower()

    for key, values in QUERY_EXPANSION.items():

        if key in query_lower:

            expanded += " " + " ".join(values)

    return expanded

In [ ]:
def reciprocal_rank_fusion(semantic_results, bm25_results, k=60):
    """
    Reciprocal Rank Fusion (RRF)

    Score(doc) = Σ 1 / (k + rank)

    Larger k -> smoother scores (60 is the standard value).
    """

    fused = {}

    # Semantic ranking
    for rank, item in enumerate(semantic_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    # BM25 ranking
    for rank, item in enumerate(bm25_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    return sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

In [ ]:
def retrieve(query, top_k=5):

    # =====================================================
    # Encode Query
    # =====================================================
    expanded_query = expand_query(query)

    query_embedding = embedding_model.encode(
        [expanded_query],
        normalize_embeddings=True
    ).astype("float32")

    # =====================================================
    # FAISS Search
    # =====================================================

    semantic_scores, semantic_indices = index.search(
        query_embedding,
        top_k * 8
    )

    semantic_results = []

    for rank, (score, idx) in enumerate(
        zip(semantic_scores[0], semantic_indices[0]),
        start=1
    ):

        semantic_results.append({

            "chunk_id": idx,
            "rank": rank,
            "semantic_score": float(score)

        })

    # =====================================================
    # BM25 Search
    # =====================================================

    tokenized_query = expanded_query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_ranked = sorted(

        enumerate(bm25_scores),

        key=lambda x: x[1],

        reverse=True

    )[:top_k * 8]

    bm25_results = []

    for rank, (idx, score) in enumerate(
        bm25_ranked,
        start=1
    ):

        bm25_results.append({

            "chunk_id": idx,
            "rank": rank,
            "bm25_score": float(score)

        })

    # =====================================================
    # Reciprocal Rank Fusion (RRF)
    # =====================================================

    k = 60

    rrf_scores = {}

    semantic_lookup = {}
    bm25_lookup = {}

    for item in semantic_results:

        cid = item["chunk_id"]

        semantic_lookup[cid] = item["semantic_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    for item in bm25_results:

        cid = item["chunk_id"]

        bm25_lookup[cid] = item["bm25_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    rrf_results = []

    for chunk_id, rrf_score in rrf_scores.items():

        chunk = all_chunks[chunk_id]

        rrf_results.append({

            "chunk_id": chunk_id,

            "page": chunk["page"],

            "document": chunk["document"],

            "text": chunk["text"],

            "semantic_score": semantic_lookup.get(chunk_id, 0.0),

            "bm25_score": bm25_lookup.get(chunk_id, 0.0),

            "rrf_score": rrf_score

        })

    # =====================================================
    # Intelligent Tie Breaking
    # =====================================================

    for item in rrf_results:

        item["final_score"] = (

            item["rrf_score"]

            + 0.001 * item["semantic_score"]

            + 0.001 * item["bm25_score"]

        )

    rrf_results = sorted(

        rrf_results,

        key=lambda x: x["final_score"],

        reverse=True

    )

    # =====================================================
    # Prevent Adjacent Matched Chunks
    # =====================================================

    selected = []

    for item in rrf_results:

        current = item["chunk_id"]

        if any(abs(current - x["chunk_id"]) <= 1 for x in selected):
            continue

        selected.append(item)

        if len(selected) == top_k:
            break

    # =====================================================
    # Context Expansion
    # =====================================================

    expanded = []

    visited = set()

    for rank, item in enumerate(selected, start=1):

        current = item["chunk_id"]

        for neighbour in [current - 1, current, current + 1]:

            if neighbour < 0:
                continue

            if neighbour >= len(all_chunks):
                continue

            if neighbour in visited:
                continue

            if all_chunks[neighbour]["document"] != item["document"]:
                continue

            visited.add(neighbour)

            chunk = all_chunks[neighbour]

            expanded.append({

                "chunk_id": chunk["chunk_id"],

                "page": chunk["page"],

                "document": chunk["document"],

                "text": chunk["text"],

                "retrieval_rank": rank,

                "context_neighbor": neighbour != current,

                "semantic_score": item["semantic_score"] if neighbour == current else None,

                "bm25_score": item["bm25_score"] if neighbour == current else None,

                "rrf_score": item["rrf_score"] if neighbour == current else None

            })

    return expanded

In [ ]:
def debug_retrieval(query, top_k=5):

    results = retrieve(query, top_k)

    print("=" * 90)
    print(f"QUERY : {query}")
    print("=" * 90)

    current_rank = None

    for chunk in results:

        if chunk["retrieval_rank"] != current_rank:

            current_rank = chunk["retrieval_rank"]

            print()
            print("=" * 90)
            print(f"RETRIEVAL RANK {current_rank}")
            print("=" * 90)

        print()

        if chunk["context_neighbor"]:

            print("Context Chunk")

        else:

            print("Matched Chunk")

            print(f"Semantic Score : {chunk['semantic_score']:.4f}")
            print(f"BM25 Score     : {chunk['bm25_score']:.4f}")
            if "rrf_score" in chunk:
              print(f"RRF Score      : {chunk['rrf_score']:.5f}")

        print(f"Document : {chunk['document']}")
        print(f"Page     : {chunk['page']}")

        if chunk.get("section"):
          print(f"Section  : {chunk['section']}")

        print(f"Chunk ID : {chunk['chunk_id']}")

        print("-" * 90)

        print(chunk["text"][:700])

        print()

In [ ]:
debug_retrieval("What algorithm was used for classification?")

debug_retrieval("What is GBDT?")

debug_retrieval("Gradient Boosted Decision Tree")

QUERY : What algorithm was used for classification?

RETRIEVAL RANK 1

Context Chunk
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
Chunk ID : 33
------------------------------------------------------------------------------------------
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.


Matched Chunk
Semantic Score : 0.6347
BM25 Score     : 13.7764
RRF Score      : 0.03

In [ ]:
query = "What algorithm was used for classification?"

results = retrieve(query)

# Keep only the best 2 chunks
context = "\n\n".join(
    [r["text"][:600] for r in results[:2]]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more reali

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs are combined, weighted by a learning r

In [ ]:
query = "What algorithm was used for classification?"

results = retrieve(query)

print(f"Retrieved {len(results)} chunks.")

Retrieved 15 chunks.


In [ ]:
TOP_CONTEXT = 3

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:TOP_CONTEXT]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs 

## Load the LLM

**v0.6.3 change:** swapped `TinyLlama/TinyLlama-1.1B-Chat-v1.0` for
`Qwen/Qwen2.5-3B-Instruct`. TinyLlama is a 1.1B base-ish chat model and was
the main source of weak/invented answers in the v0.6.2 eval run. Qwen2.5-3B-Instruct
is a proper instruction-tuned model that follows system-prompt rules
(including refusal instructions) far more reliably, while still being small
enough to run in fp16 on a single Colab T4/A100 GPU.

If you're on a smaller or slower GPU, drop down to
`"Qwen/Qwen2.5-1.5B-Instruct"` — same model family and prompting behavior,
about half the memory and latency, with some quality trade-off.

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch

# v0.6.3: replaced TinyLlama-1.1B with a stronger instruct model.
# Qwen2.5-3B-Instruct fits comfortably in fp16 on a single T4/A100 Colab GPU
# and follows instructions (including refusal instructions) far more
# reliably than the 1.1B model did.
#
# Too slow / out of memory on your GPU? Swap to the lighter model below —
# same family, much lighter, still meaningfully stronger than TinyLlama:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Prevent generation warnings
model.generation_config.max_length = None
model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"{MODEL_NAME} Loaded")


Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-3B-Instruct Loaded


## Build the prompt

**v0.6.3 change:** `build_prompt` becomes `build_system_prompt` — it now
returns only the *system* portion (rules + conversation history + context).
The question is passed separately as the user turn, and the two are combined
via `tokenizer.apply_chat_template(...)` in the generation cell below. Raw
f-string prompts like the old `build_prompt` work poorly with instruct
models, which are trained on a specific chat format — using it properly is
a big part of why refusal behavior improves in this version.

In [ ]:
def build_system_prompt(context):

    history = get_chat_history()

    return f"""You are an AI assistant answering questions from a research report.

Rules:

- Answer ONLY using the Context.
- Do NOT use outside knowledge.
- Do NOT invent facts or names.
- If the answer is not present in the context, reply exactly:

I could not find that information in the provided document.

- Keep answers concise.
- Maximum 3 sentences.

Previous Conversation:
{history}

Context:
{context}"""


In [ ]:
# ----------------------------
# Conversation Memory
# ----------------------------

conversation_history = []


def get_chat_history(max_turns=3):
    """
    Returns the previous conversation history
    as formatted text.
    """

    if len(conversation_history) == 0:
        return ""

    history = ""

    for q, a in conversation_history[-max_turns:]:

        history += f"User: {q}\n"
        history += f"Assistant: {a}\n\n"

    return history

## Generate an answer

**v0.6.3 changes:**
- Uses `tokenizer.apply_chat_template(...)` to format the system prompt +
  question the way Qwen2.5-Instruct actually expects, instead of a raw
  completion string.
- Decodes only the newly generated tokens
  (`outputs[0][inputs["input_ids"].shape[1]:]`) instead of slicing the
  decoded string by `len(prompt)`. The old approach can silently misalign
  when the tokenizer's string round-trip doesn't exactly match the original
  prompt text (e.g. special tokens) — decoding from the token boundary is
  the robust way to do this.

In [ ]:
def generate_answer(question, context):

    system_prompt = build_system_prompt(context)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        do_sample=False,
        repetition_penalty=1.15,
        max_new_tokens=150,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    # Decode only the newly generated tokens (robust against
    # tokenizer round-trip mismatches on the prompt half).
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # Prevent the model from continuing into a fake next turn
    stop_words = [
        "\nQuestion:",
        "\nContext:",
        "\nUser:",
        "\nAssistant:"
    ]

    for stop in stop_words:
        if stop in answer:
            answer = answer.split(stop)[0].strip()

    return answer


Quick raw-generation smoke test (this bypasses the confidence gate below — useful for checking the LLM swap works before wiring in the don't-know mode).

In [ ]:
query = "What algorithm was used for classification?"

results = retrieve(query)

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:3]
)

answer = generate_answer(query, context)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print("ANSWER")
print(answer)

QUESTION
What algorithm was used for classification?

ANSWER
Gradient Boosted Decision Trees (GBDT) were used for classification.


## Retrieval Confidence Threshold ("Don't Know" Mode)

This is the core addition in v0.6.3. Up to now, refusing to answer
out-of-document questions relied entirely on the prompt instruction — and
the v0.6.2 eval run showed the LLM doesn't reliably obey that instruction
(it invented an optimizer and a GPU model for questions the document never
answers).

The fix: check retrieval confidence **before** calling the LLM at all. If
the best-matching retrieved chunk isn't actually relevant, there's no need
to ask the LLM to "please don't make something up" — just don't generate.

`answer_question()` becomes the new single entry point for asking a
question; it wraps `retrieve()` + the confidence check + `generate_answer()`
into one call, and is now used by both the interactive loop and the eval
pipeline below, so the gold evaluation actually exercises this gate.

In [ ]:
# ==========================================================
# Retrieval Confidence Threshold ("Don't Know" Mode)
# ==========================================================

# Placeholder default — DO NOT trust this number as-is.
# Run calibrate_confidence_threshold() once gold_eval.json is loaded
# (further down this notebook) and set this based on the actual score
# distribution it prints, then re-run this cell.
CONFIDENCE_THRESHOLD = 0.35

DONT_KNOW_MESSAGE = "I could not find that information in the provided document."


def assess_confidence(retrieved_chunks):
    """
    Confidence = the best semantic similarity score among the top-ranked
    (non-neighbor) retrieved chunks.

    Context-expansion neighbor chunks don't carry their own semantic_score
    (it's set to None in retrieve()), so only chunks that were actually
    matched by retrieval are considered.
    """

    scores = [
        chunk["semantic_score"]
        for chunk in retrieved_chunks
        if chunk["semantic_score"] is not None
    ]

    if not scores:
        return 0.0

    return max(scores)


def answer_question(question, retrieved_chunks=None, top_context=3):
    """
    Single entry point for asking a question end-to-end:

    1. Retrieve chunks (or reuse chunks already retrieved by the caller).
    2. Check retrieval confidence BEFORE calling the LLM.
    3. If confidence is below CONFIDENCE_THRESHOLD, skip generation
       entirely and return the don't-know message — cheaper and far
       more reliable than trusting the LLM to refuse on its own.
    4. Otherwise generate a grounded answer as before.

    Returns (answer, confidence, retrieved_chunks).
    """

    if retrieved_chunks is None:
        retrieved_chunks = retrieve(question)

    confidence = assess_confidence(retrieved_chunks)

    if confidence < CONFIDENCE_THRESHOLD:
        return DONT_KNOW_MESSAGE, confidence, retrieved_chunks

    context = "\n\n".join(
        chunk["text"] for chunk in retrieved_chunks[:top_context]
    )

    answer = generate_answer(question, context)

    return answer, confidence, retrieved_chunks


def calibrate_confidence_threshold():
    """
    Runs retrieval confidence over every question in gold_eval.json and
    prints the score, split by answerable / unanswerable, so you can pick
    a CONFIDENCE_THRESHOLD based on real numbers instead of a guess.

    Requires `gold` to already be loaded (see the gold_eval.json cell
    further down this notebook) — call this after that cell runs.
    """

    answerable_scores = []
    unanswerable_scores = []

    for sample in gold:

        results = retrieve(sample["question"])
        confidence = assess_confidence(results)

        if sample["answerable"]:
            answerable_scores.append(confidence)
        else:
            unanswerable_scores.append(confidence)

        label = "ANSWERABLE" if sample["answerable"] else "UNANSWERABLE"
        print(f"[{label:>12}] {confidence:.3f}  {sample['question']}")

    print("\n" + "=" * 70)

    if answerable_scores:
        print(
            f"Answerable   -> min {min(answerable_scores):.3f}, "
            f"max {max(answerable_scores):.3f}, "
            f"avg {sum(answerable_scores)/len(answerable_scores):.3f}"
        )

    if unanswerable_scores:
        print(
            f"Unanswerable -> min {min(unanswerable_scores):.3f}, "
            f"max {max(unanswerable_scores):.3f}, "
            f"avg {sum(unanswerable_scores)/len(unanswerable_scores):.3f}"
        )

    print("=" * 70)
    print("Pick CONFIDENCE_THRESHOLD between the top of the unanswerable")
    print("range and the bottom of the answerable range. If the ranges")
    print("overlap, there's no single clean threshold yet — that's a real")
    print("finding, not a bug in this function.")


Interactive loop, now routed through `answer_question()` so the confidence gate is active.

In [ ]:
while True:
    print("\n" + "=" * 80)

    question = input("\nAsk a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("\nGoodbye!")
        break

    answer, confidence, _ = answer_question(question)

    conversation_history.append((question, answer))

    print("\n" + "=" * 80)
    print(f"Confidence: {confidence:.3f}")
    print(answer)




Ask a question (type 'exit' to quit): How was the dataset split?

Confidence: 0.539
The dataset was split into a training set (90% of the data) and a validation set (10%) using stratified sampling to maintain the same proportions of different classes. A difficulty-aware splitting strategy was then applied: multiple random partitions were created with stratification, misclassifications identified, and these helped define "easy" and "hard" subsets. Finally, balanced sampling ensured proportional representation in both training and validation sets.


Ask a question (type 'exit' to quit): What's the capital of France?

Confidence: 0.093
I could not find that information in the provided document.


Ask a question (type 'exit' to quit): What's your favorite movie?

Confidence: 0.132
I could not find that information in the provided document.


Ask a question (type 'exit' to quit): What GPU was used for training?

Confidence: 0.301
I could not find that information in the provided document.

## Known Limitations (v0.6.3)

This version replaces TinyLlama with Qwen2.5-3B-Instruct and adds a
retrieval-confidence "don't know" gate, so refusal no longer depends solely
on the LLM following instructions.

Remaining limitations:

- `CONFIDENCE_THRESHOLD` is a single global number tuned against one
  10-question gold set on one document — treat it as a starting point, not
  a final value. Re-run `calibrate_confidence_threshold()` as the gold set
  grows (v0.6.2's planned 50-100 question set) or as new documents are added.
- The gate uses semantic similarity only; it doesn't yet incorporate the
  BM25 or RRF score, which could catch cases semantic search misses.
- Confidence is computed from the retrieved chunks only — it doesn't (yet)
  check whether the *generated answer* is actually grounded in the context
  it was given. That's a reasonable v0.7+ addition (answer verification /
  a second grounding check) but out of scope here.
- Still evaluated on a single document; behavior on structurally different
  PDFs (scanned, multi-column, no clear headings) is untested.


Demo: an out-of-domain question. This is exactly the case the confidence
gate is for — nothing in the gravitational-wave report is about sports, so
retrieval confidence should be low and this should return the don't-know
message *without the LLM ever being called*, rather than relying on the
LLM to notice on its own.

In [ ]:
query = "Which sport is most played?"

answer, confidence, retrieved_chunks = answer_question(query)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print(f"Confidence: {confidence:.3f}")
print("ANSWER")
print(answer)


QUESTION
Which sport is most played?

Confidence: 0.070
ANSWER
I could not find that information in the provided document.


###Gold Evaluation Set

In [ ]:
questions = [
    "What algorithm was used for classification?",
    "How was the dataset split?",
    "Why was GBDT chosen?",
    "Which inspiral parameters were used?",
    "What does Classifier A predict?",
    "Who is the author of the report?",
    "What is SHAP used for?",
    "Which real gravitational-wave events were analysed?",
    "What optimizer was used to train the neural network?",
    "What GPU was used for training?"
]

In [ ]:
for q in questions:
    debug_retrieval(q)

QUERY : What algorithm was used for classification?

RETRIEVAL RANK 1

Context Chunk
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
Chunk ID : 33
------------------------------------------------------------------------------------------
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.


Matched Chunk
Semantic Score : 0.6347
BM25 Score     : 13.7764
RRF Score      : 0.03

In [ ]:
import json

In [ ]:
with open("gold_eval.json") as f:
    gold = json.load(f)

Now that `gold` is loaded, calibrate the confidence threshold against it
before running the full evaluation. Look at the printed ranges, update
`CONFIDENCE_THRESHOLD` in the cell above if needed, and re-run that cell.

In [ ]:
calibrate_confidence_threshold()


[  ANSWERABLE] 0.635  What algorithm was used for classification?
[  ANSWERABLE] 0.539  How was the dataset split?
[  ANSWERABLE] 0.588  Why was GBDT chosen?
[  ANSWERABLE] 0.411  Which inspiral parameters were used?
[  ANSWERABLE] 0.472  What does Classifier A predict?
[  ANSWERABLE] 0.724  Who is the author of Machine Learning Classification of Binary Neutron Star Remnants Using Gravitational Wave Data?
[  ANSWERABLE] 0.475  What is SHAP used for?
[  ANSWERABLE] 0.431  Which real gravitational-wave events were analysed?
[UNANSWERABLE] 0.277  What optimizer was used to train the neural network?
[UNANSWERABLE] 0.301  What GPU was used for training?

Answerable   -> min 0.411, max 0.724, avg 0.534
Unanswerable -> min 0.277, max 0.301, avg 0.289
Pick CONFIDENCE_THRESHOLD between the top of the unanswerable
range and the bottom of the answerable range. If the ranges
overlap, there's no single clean threshold yet — that's a real
finding, not a bug in this function.


In [ ]:
def evaluate_retrieval(retrieved_chunks, sample):

    if not sample["answerable"]:
        return True

    retrieved_pages = [
        chunk["page"]
        for chunk in retrieved_chunks[:3]
    ]

    retrieved_chunk_ids = [
        chunk["chunk_id"]
        for chunk in retrieved_chunks[:3]
    ]

    passed = (
        sample["gold_chunk"] in retrieved_chunk_ids
        or
        sample["gold_page"] in retrieved_pages
    )

    print("Retrieval :", "PASS" if passed else "FAIL")
    print("Expected Page :", sample["gold_page"])
    print("Retrieved Pages :", retrieved_pages)

    return passed

**v0.6.3 change:** now calls `answer_question()` instead of `generate_answer()` directly, so the gold evaluation actually exercises the confidence gate, not just raw generation.

In [ ]:
def evaluate_generation(sample, retrieved_chunks):

    answer, confidence, _ = answer_question(
        sample["question"],
        retrieved_chunks=retrieved_chunks
    )

    print(f"\nRetrieval Confidence: {confidence:.3f}")

    print("\nGenerated Answer\n")
    print(answer)

    return answer


In [ ]:
def evaluate_answer(answer, sample):

    if not sample["answerable"]:

        passed = (
            "could not find" in answer.lower()
            or
            "not found" in answer.lower()
        )

    else:

        hits = sum(
            keyword.lower() in answer.lower()
            for keyword in sample["keywords"]
        )

        passed = (
            hits / len(sample["keywords"])
        ) >= 0.65

    print("\nExpected Answer\n")
    print(sample["expected_answer"])

    print("\nGeneration :", "PASS" if passed else "FAIL")

    return passed

In [ ]:
def evaluate_rag():

    retrieval_correct = 0
    generation_correct = 0

    total = len(gold)

    for sample in gold:

        print("=" * 80)
        print("QUESTION")
        print(sample["question"])
        print("=" * 80)

        retrieved_chunks = retrieve(sample["question"])

        retrieval_pass = evaluate_retrieval(
            retrieved_chunks,
            sample
        )

        if retrieval_pass:
            retrieval_correct += 1

        answer = evaluate_generation(
            sample,
            retrieved_chunks
        )

        generation_pass = evaluate_answer(
            answer,
            sample
        )

        if generation_pass:
            generation_correct += 1

    print("\n" + "=" * 80)

    print(f"Retrieval Accuracy : {retrieval_correct}/{total} ({100*retrieval_correct/total:.1f}%)")
    print(f"Generation Accuracy: {generation_correct}/{total} ({100*generation_correct/total:.1f}%)")

In [ ]:
def evaluate_subset(indices):

    retrieval_correct = 0
    generation_correct = 0

    for idx in indices:

        sample = gold[idx]

        print("=" * 80)
        print("QUESTION")
        print(sample["question"])
        print("=" * 80)

        retrieved_chunks = retrieve(sample["question"])

        retrieval_pass = evaluate_retrieval(
            retrieved_chunks,
            sample
        )

        if retrieval_pass:
            retrieval_correct += 1

        answer = evaluate_generation(
            sample,
            retrieved_chunks
        )

        generation_pass = evaluate_answer(
            answer,
            sample
        )

        if generation_pass:
            generation_correct += 1

    print("\n" + "=" * 80)

    print(
        f"Retrieval Accuracy : {retrieval_correct}/{len(indices)} "
        f"({100*retrieval_correct/len(indices):.1f}%)"
    )

    print(
        f"Generation Accuracy: {generation_correct}/{len(indices)} "
        f"({100*generation_correct/len(indices):.1f}%)"
    )

In [ ]:

evaluate_subset([6])

QUESTION
What is SHAP used for?
Retrieval : PASS
Expected Page : 9
Retrieved Pages : [8, 9, 9]

Retrieval Confidence: 0.475

Generated Answer

SHAP was used to interpret model predictions and rank feature importance by measuring the marginal contribution of each feature.

Expected Answer

To interpret model predictions and rank feature importance.

Generation : PASS

Retrieval Accuracy : 1/1 (100.0%)
Generation Accuracy: 1/1 (100.0%)


In [ ]:
evaluate_rag()

QUESTION
What algorithm was used for classification?
Retrieval : PASS
Expected Page : 8
Retrieved Pages : [8, 8, 8]

Retrieval Confidence: 0.635

Generated Answer

Gradient Boosted Decision Trees (GBDT) were used for classification.

Expected Answer

Gradient Boosted Decision Trees (GBDT).

Generation : PASS
QUESTION
How was the dataset split?
Retrieval : PASS
Expected Page : 8
Retrieved Pages : [7, 8, 8]

Retrieval Confidence: 0.539

Generated Answer

The dataset was split into a training set with 90% of the data and a validation set with 10%, using stratified sampling to maintain the same proportions of different classes. A difficulty-aware splitting strategy was then applied to ensure the validation set represents the overall complexity of the dataset better.

Expected Answer

90% training and 10% validation using stratified, difficulty-aware splitting.

Generation : PASS
QUESTION
Why was GBDT chosen?
Retrieval : PASS
Expected Page : 8
Retrieved Pages : [8, 8, 8]

Retrieval Confiden